# A7 synthetic evaluation (SYNTHETIC: TTS audio, no child speech)

Runtime: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. About 10-15 minutes.

1. Clones the repo and installs the scorer (`ai[model]`).
2. Renders the 154 spoken texts of `ai/eval/synthetic-set.json` with Google TTS (gTTS; Microsoft edge-tts is blocked in Colab).
3. Scores every case with the real scorer and compares the result with the injected errors.
4. Scores the item bank's own Azure Maryam clips against their text (every flag there is a false alarm).
5. Prints the report and downloads `a7-results.zip`. Send that zip (or paste the printed report) back.

In [ ]:
%%capture
BRANCH = "feat/a7-synthetic-eval"
!rm -rf Kuwait-Dyslexia && git clone -q -b $BRANCH https://github.com/DXstr/Kuwait-Dyslexia.git
!pip -q install -U gTTS
!pip -q install -e "Kuwait-Dyslexia/ai[model]"


In [ ]:
import os, sys, json, time, platform, hashlib, datetime, pathlib
os.chdir("/content/Kuwait-Dyslexia")
sys.path.insert(0, "ai")
import torch
from basira_ai.acoustic import Listener, load_audio
from basira_ai.contracts_gen import ScoreItemRequestMeta
from basira_ai.eval_synthetic import evaluate_case, summarize, summarize_clips, report_markdown
from basira_ai.reference import reference_data
from basira_ai.pronunciation import routing
REF = reference_data()
SET = json.loads(pathlib.Path("ai/eval/synthetic-set.json").read_text(encoding="utf-8"))
BANK = json.loads(pathlib.Path("content/dist/item-bank.json").read_text(encoding="utf-8"))
device = "cuda" if torch.cuda.is_available() else "cpu"
listener = Listener(device=device)
print(len(SET["cases"]), "cases; device", device)

In [ ]:
# 2. TTS of every distinct spoken text (cached on disk; retries because gTTS rate-limits)
from gtts import gTTS
audio_dir = pathlib.Path("/content/a7_tts"); audio_dir.mkdir(exist_ok=True)
def tts_path(text):
    p = audio_dir / (hashlib.sha1(text.encode()).hexdigest()[:12] + ".mp3")
    for attempt in range(5):
        if p.exists() and p.stat().st_size > 0:
            return p
        try:
            gTTS(text, lang="ar").save(str(p)); time.sleep(0.3)
        except Exception as e:
            print("retry", attempt, e); time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"TTS failed for {text}")
texts = sorted({c["spoken_text"] for c in SET["cases"]})
for i, t in enumerate(texts):
    tts_path(t)
    if i % 25 == 0: print(i, "/", len(texts))
print("TTS done:", len(texts))

In [ ]:
# 3. score every case with the real scorer
def score(wav, item_id, item_type, text, lexicon):
    meta = ScoreItemRequestMeta.model_validate({"item_id": item_id, "item_type": item_type, "expected_text": text,
                                                "dialect_lexicon_id": lexicon})
    return listener.score_item(wav, meta, synthetic=True).model_dump(mode="json", by_alias=True)
wav_cache, results, failures = {}, [], []
for k, c in enumerate(SET["cases"]):
    try:
        wav = wav_cache.setdefault(c["spoken_text"], load_audio(str(tts_path(c["spoken_text"]))))
        r = evaluate_case(c, score(wav, c["item_id"], "real_word", c["expected_text"], c["lexicon"]), REF)
        r["spoken_text"], r["expected_text"] = c["spoken_text"], c["expected_text"]
        results.append(r)
    except Exception as e:
        failures.append({"case_id": c["case_id"], "error": repr(e)})
    if k % 50 == 0: print(k, "/", len(SET["cases"]))
print("scored", len(results), "failed", len(failures))

In [ ]:
# 4. the item bank's own Maryam clips, scored against their text (real words + pseudowords)
clip_results, clip_failures = [], []
for item in BANK["items"]:
    if item["type"] not in ("real_word", "pseudoword") or not item.get("audio_path"):
        continue
    try:
        wav = load_audio(str(pathlib.Path("content/dist") / item["audio_path"]))
        resp = score(wav, item["id"], item["type"], item["text"], "msa")
        case = {"case_id": item["id"], "expected_text": item["text"], "severity": "none", "lexicon": "msa",
                "dialect_cell": None, "injected": []}
        r = evaluate_case(case, resp, REF); r["expected_text"] = item["text"]; clip_results.append(r)
    except Exception as e:
        clip_failures.append({"item_id": item["id"], "error": repr(e)})
print("clips scored", len(clip_results), "failed", len(clip_failures))

In [ ]:
# 5. report + download
summary, clips = summarize(results), summarize_clips(clip_results)
meta = {"set_version": SET["version"], "seed": SET["seed"], "model_version": "; ".join(routing()[m]["id"] for m in ("phone_model", "letter_model")),
        "run_at": datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%MZ"),
        "runtime": f"Colab {device} {torch.cuda.get_device_name(0) if device == 'cuda' else ''}, torch {torch.__version__}, Python {platform.python_version()}",
        "tts": "gTTS (Google Translate TTS, lang=ar)"}
report = report_markdown(summary, clips, meta)
print(report)
out = pathlib.Path("/content/a7-results"); out.mkdir(exist_ok=True)
(out / "report-synthetic.md").write_text(report, encoding="utf-8")
(out / "results-synthetic.json").write_text(json.dumps({"synthetic": True, "meta": meta, "summary": summary,
    "clips_summary": clips, "cases": results, "clips": clip_results, "failures": failures,
    "clip_failures": clip_failures}, ensure_ascii=False, indent=1), encoding="utf-8")
!cd /content && zip -qr a7-results.zip a7-results
from google.colab import files
files.download("/content/a7-results.zip")